In [ ]:
# Contributors: Antonio Meneses, Julián Consuegra, Juan Pablo Miró-Quesada, Tomás Roschge
# Course: Recommendation Engines · IE University · 2025-26
# Notebook 01: Non-Personalized Recommender

In [ ]:
import sys, os

_nb_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.path.join(_nb_dir, '..'),
    os.path.join(os.getcwd(), '..'),
    '/content/group5-rec-engines',
    os.getcwd(),
]
for _root in _candidates:
    if os.path.isfile(os.path.join(_root, 'src', 'utils.py')):
        sys.path.insert(0, os.path.join(_root, 'src'))
        os.chdir(os.path.join(_root, 'notebooks'))
        break
else:
    raise FileNotFoundError("Cannot find src/utils.py. Run from the repo root or notebooks/ directory.")

from utils import *

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

## 1. Data Loading & Train/Test Split

In [ ]:
df, df_meta = load_data()
print(f'Reviews loaded: {len(df):,}')
print(f'Products loaded: {len(df_meta):,}')

In [ ]:
train_df, test_df, split_info = temporal_train_test_split(df)
train_items = split_info['train_items']

In [ ]:
lookups = build_lookup_structures(train_df, test_df)
GLOBAL_MEAN       = lookups['global_mean']
train_user_items  = lookups['train_user_items']
all_items         = lookups['all_items']

sample_users = sample_eval_users(lookups, n=1000)
print(f'Global mean (train): {GLOBAL_MEAN:.4f}')

## 2. Non-Personalized Recommenders

Three non-personalized baselines that ignore individual preferences:

1. **Global Mean** — predicts the overall average rating for every user-item pair.
2. **Item Mean** — predicts each item's historical average (falls back to global mean for cold-start items).
3. **Damped Mean (Bayesian Average)** — `score = (n * item_avg + m * global_avg) / (n + m)` with
   damping factor `m=50`. This shrinks items with few ratings toward the global mean, preventing
   a single 5-star review from dominating the popularity ranking.

In [ ]:
# Rating prediction — vectorized
item_means = train_df.groupby('item_id')['rating'].mean()
counts     = train_df['item_id'].value_counts()
sums       = train_df.groupby('item_id')['rating'].sum()
damped_means = (sums + DAMPING_FACTOR * GLOBAL_MEAN) / (counts + DAMPING_FACTOR)

item_pred   = test_df['item_id'].map(item_means).fillna(GLOBAL_MEAN)
damped_pred = test_df['item_id'].map(damped_means).fillna(GLOBAL_MEAN)
global_pred = pd.Series([GLOBAL_MEAN] * len(test_df), index=test_df.index)

rmse_gm = rmse(test_df['rating'], global_pred)
mae_gm  = mae(test_df['rating'], global_pred)
rmse_im = rmse(test_df['rating'], item_pred)
mae_im  = mae(test_df['rating'], item_pred)
rmse_dm = rmse(test_df['rating'], damped_pred)
mae_dm  = mae(test_df['rating'], damped_pred)

print(f'Global Mean  — RMSE: {rmse_gm:.4f}, MAE: {mae_gm:.4f}')
print(f'Item Mean    — RMSE: {rmse_im:.4f}, MAE: {mae_im:.4f}')
print(f'Damped Mean  — RMSE: {rmse_dm:.4f}, MAE: {mae_dm:.4f}')

## 3. Ranking Evaluation

In [ ]:
# Precompute top-200 item lists — same for all users (non-personalized)
top_items_mean   = list(item_means.sort_values(ascending=False).head(200).index)
top_items_damped = list(damped_means.sort_values(ascending=False).head(200).index)

def recommend_np(user_id, ranked_list):
    seen = train_user_items.get(user_id, set())
    return [it for it in ranked_list if it not in seen][:K]

In [ ]:
results_np = {}
recs_np    = {}   # saved for diversity computation in notebook 05

configs = [
    ('Global Mean', rmse_gm, mae_gm,  top_items_damped),
    ('Item Mean',   rmse_im, mae_im,   top_items_mean),
    ('Damped Mean', rmse_dm, mae_dm,   top_items_damped),
]

for name, r_val, m_val, ranked_list in configs:
    fn = lambda uid, rl=ranked_list: recommend_np(uid, rl)
    all_recs, metrics = evaluate_ranking(fn, sample_users, lookups)
    recs_np[name] = {u: r for u, r in zip(sample_users, all_recs)}
    results_np[name] = {'RMSE': r_val, 'MAE': m_val, 'Diversity': None, **metrics}
    print_metrics(name, r_val, m_val, metrics)

## 4. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Non-Personalized Recommenders', fontsize=13, fontweight='bold')

# Rating distribution of top-200 damped items
top_items_series = damped_means.sort_values(ascending=False).head(200)
pop_counts = counts[top_items_series.index]
axes[0].hist(pop_counts.values, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Review Counts — Top-200 Popularity Items')
axes[0].set_xlabel('Number of reviews')
axes[0].set_ylabel('Items')

# RMSE comparison
names  = ['Global\nMean', 'Item\nMean', 'Damped\nMean']
rmses  = [rmse_gm, rmse_im, rmse_dm]
colors = ['gray', 'steelblue', 'coral']
axes[1].bar(names, rmses, color=colors, edgecolor='white')
axes[1].set_title('RMSE Comparison — Non-Personalized Models')
axes[1].set_ylabel('RMSE')
axes[1].set_ylim(min(rmses) - 0.02, max(rmses) + 0.02)

plt.tight_layout()
plt.show()

## 5. Save Results

In [ ]:
output = {
    'results_np':        results_np,
    'item_means':        item_means.to_dict(),
    'damped_means':      damped_means.to_dict(),
    'top_items_mean':    top_items_mean,
    'top_items_damped':  top_items_damped,
    'recs_np':           recs_np,           # {model_name: {user_id: [item_ids]}}
    'sample_users':      sample_users,
}

os.makedirs('../results', exist_ok=True)
with open('../results/01_non_personalized.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Saved: ../results/01_non_personalized.pkl')
print('Keys:', list(output.keys()))